# Utsunomiya & Haga — Location Overview Map (ArcGIS API)

Audience-facing map of **Utsunomiya City** and adjacent **Haga Town** (Tochigi Prefecture, Japan) with:

- Satellite / **Sentinel-2**-style imagery (clear-sky optical basemap)
- Municipal boundaries
- **JR Utsunomiya Station** (red dot)
- **JR railway** corridor (blue line)
- **Utsunomiya LRT / Lightline** (yellow line)
- **清原工業団地 (Kiyohara Industrial Park)** (orange circle)

Run all cells top-to-bottom; the interactive map appears in the last cell.

In [ ]:
# Install ArcGIS API dependencies if needed (safe to re-run)
%pip install -q arcgis arcgis-mapping geopandas pyogrio shapely

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely.geometry import LineString, Point

from arcgis.features import GeoAccessor
from arcgis.gis import GIS
from arcgis.map import Map, renderers, symbols
from arcgis.raster import ImageryLayer

DATA_DIR = Path("data")
GADM_CACHE = Path("data/gadm41_JPN_2.json")
LRT_STOPS_PATH = DATA_DIR / "lrt_stops.shp"

## Connect to ArcGIS

Uses an anonymous session by default. For ArcGIS Online organization features, set credentials via environment variables or uncomment and edit the `GIS(...)` line below (same pattern as `Travel_LRT.ipynb`).

In [ ]:
import os

arcgis_user = os.environ.get("ARCGIS_USERNAME")
arcgis_password = os.environ.get("ARCGIS_PASSWORD")

if arcgis_user and arcgis_password:
    gis = GIS(username=arcgis_user, password=arcgis_password)
else:
    gis = GIS()  # anonymous — sufficient for basemap + Living Atlas Sentinel-2

print("Connected:", gis.url)

## Prepare map layers

Administrative boundaries are loaded from GADM (level 2). The LRT alignment follows official stop order in `data/lrt_stops.shp`. JR geometry is a representative main-line segment through JR Utsunomiya Station for visualization.

In [ ]:
GADM_URL = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_JPN_2.json"

if not GADM_CACHE.exists():
    GADM_CACHE.parent.mkdir(parents=True, exist_ok=True)
    import urllib.request

    print("Downloading GADM Japan municipalities …")
    urllib.request.urlretrieve(GADM_URL, GADM_CACHE)

adm2 = gpd.read_file(GADM_CACHE)
municipalities = adm2[adm2["NAME_2"].isin(["Utsunomiya", "Haga"])].copy()
municipalities["municipality"] = municipalities["NAME_2"]
municipalities = municipalities[["municipality", "geometry"]]
print(municipalities[["municipality"]])

In [ ]:
# JR Utsunomiya Station (main station, west side of city center)
JR_UTSUNOMIYA = {"name": "JR宇都宮駅", "lon": 139.898331, "lat": 36.559083}

jr_station = gpd.GeoDataFrame(
    {"label": [JR_UTSUNOMIYA["name"]]},
    geometry=[Point(JR_UTSUNOMIYA["lon"], JR_UTSUNOMIYA["lat"])],
    crs="EPSG:4326",
)

# Representative JR main line (Tohoku corridor) through Utsunomiya — visualization only
jr_line_coords = [
    (139.89815, 36.5285),
    (139.89825, 36.5420),
    (139.89833, 36.55908),
    (139.89845, 36.5755),
    (139.89855, 36.5920),
    (139.89870, 36.6085),
]
jr_line = gpd.GeoDataFrame(
    {"label": ["JR東北本線（代表区間）"]},
    geometry=[LineString(jr_line_coords)],
    crs="EPSG:4326",
)

# Kiyohara Industrial Park (清原工業団地) — approximate park extent
KIYOHARA_CENTER = Point(139.98939, 36.544502)
kiyohara_circle = gpd.GeoDataFrame(
    {"label": ["清原工業団地"]},
    geometry=[KIYOHARA_CENTER.buffer(0.022)],  # ~2.4 km radius at this latitude
    crs="EPSG:4326",
)

# Utsunomiya LRT / Lightline — stop sequence from project shapefile
LRT_ROUTE_ORDER = [
    "宇都宮駅東口",
    "東宿郷",
    "駅東公園前",
    "峰",
    "陽東3丁目",
    "宇都宮大学陽東キャンパス",
    "平石",
    "平石中央小学校前",
    "飛山城跡",
    "清陵高校前",
    "清原地区市民センター前",
    "グリーンスタジアム前",
    "ゆいの杜西",
    "ゆいの杜中央",
    "ゆいの杜東",
    "芳賀台",
    "芳賀町工業団地管理センター前",
    "かしの森公園前",
    "芳賀・高根沢工業団地",
]

lrt_stops = gpd.read_file(LRT_STOPS_PATH)
if lrt_stops.crs is None:
    lrt_stops = lrt_stops.set_crs("EPSG:4326")
elif lrt_stops.crs.to_epsg() != 4326:
    lrt_stops = lrt_stops.to_crs("EPSG:4326")

lrt_coords = []
for stop_name in LRT_ROUTE_ORDER:
    row = lrt_stops.loc[lrt_stops["stop_name"] == stop_name].iloc[0]
    lrt_coords.append((float(row.stop_lon), float(row.stop_lat)))

lrt_line = gpd.GeoDataFrame(
    {"label": ["宇都宮芳賀ライトレール線"]},
    geometry=[LineString(lrt_coords)],
    crs="EPSG:4326",
)

study_bounds = municipalities.total_bounds
print(f"Study extent (WGS84): {study_bounds}")

## Build the interactive map

- **Basemap:** Esri World Imagery (high-resolution optical imagery, typically clear weather)
- **Overlay:** Esri Living Atlas **Sentinel-2 Views** (semi-transparent) for Sentinel-2-class multispectral context

In [ ]:
m = Map(gis=gis, location="Utsunomiya, Tochigi, Japan")
m.basemap.basemap = "satellite"

# Sentinel-2 Views (Living Atlas) — similar multispectral satellite family
s2_items = gis.content.search("Sentinel-2 Views", item_type="Image Service", max_items=1)
if s2_items:
    s2_layer = ImageryLayer(s2_items[0].url, gis=gis)
    m.content.add(s2_layer, options={"title": "Sentinel-2 Views (reference)", "opacity": 0.25})

m.legend.enabled = True
m.layer_list.enabled = True

In [ ]:
# Symbol / renderer definitions
boundary_renderer = renderers.SimpleRenderer(
    symbol=symbols.SimpleFillSymbolEsriSFS(
        style=symbols.SimpleFillSymbolStyle.esri_sfs_solid,
        color=[255, 255, 255, 40],
        outline=symbols.SimpleLineSymbolEsriSLS(color=[255, 255, 255, 230], width=2),
    )
)

jr_station_renderer = renderers.SimpleRenderer(
    symbol=symbols.SimpleMarkerSymbolEsriSMS(
        style=symbols.SimpleMarkerSymbolStyle.esri_sms_circle,
        color=[220, 20, 60, 255],
        size=14,
        outline=symbols.SimpleLineSymbolEsriSLS(color=[255, 255, 255, 255], width=1.5),
    )
)

jr_line_renderer = renderers.SimpleRenderer(
    symbol=symbols.SimpleLineSymbolEsriSLS(color=[30, 144, 255, 255], width=4)
)

lrt_line_renderer = renderers.SimpleRenderer(
    symbol=symbols.SimpleLineSymbolEsriSLS(color=[255, 255, 0, 255], width=5)
)

kiyohara_renderer = renderers.SimpleRenderer(
    symbol=symbols.SimpleFillSymbolEsriSFS(
        style=symbols.SimpleFillSymbolStyle.esri_sfs_null,
        color=[0, 0, 0, 0],
        outline=symbols.SimpleLineSymbolEsriSLS(color=[255, 140, 0, 255], width=3),
    )
)

# Add layers (bottom → top)
GeoAccessor.from_geodataframe(municipalities).spatial.plot(
    map_widget=m,
    name="Utsunomiya & Haga boundaries",
    renderer=boundary_renderer,
)

GeoAccessor.from_geodataframe(jr_line).spatial.plot(
    map_widget=m,
    name="JR line (representative)",
    renderer=jr_line_renderer,
)

GeoAccessor.from_geodataframe(lrt_line).spatial.plot(
    map_widget=m,
    name="LRT / Lightline",
    renderer=lrt_line_renderer,
)

GeoAccessor.from_geodataframe(kiyohara_circle).spatial.plot(
    map_widget=m,
    name="Kiyohara Industrial Park (清原工業団地)",
    renderer=kiyohara_renderer,
)

GeoAccessor.from_geodataframe(jr_station).spatial.plot(
    map_widget=m,
    name="JR Utsunomiya Station",
    renderer=jr_station_renderer,
)

pad = 0.03
xmin, ymin, xmax, ymax = study_bounds
m.extent = {
    "xmin": xmin - pad,
    "ymin": ymin - pad,
    "xmax": xmax + pad,
    "ymax": ymax + pad,
    "spatialReference": {"wkid": 4326},
}

print("Layers on map:")
for layer in m.content.layers:
    print(" -", getattr(layer, "title", layer))

## Display map

Pan/zoom in the widget below. Toggle layers in the layer list. For a static shareable file, run the optional export cell after this one.

In [ ]:
m

In [ ]:
# Optional: export standalone HTML (requires Jupyter display context)
out_dir = Path("outputs/figures")
out_dir.mkdir(parents=True, exist_ok=True)
html_path = out_dir / "utsunomiya_location_overview_map.html"
m.export_to_html(str(html_path))
print(f"Saved: {html_path}")